In [ ]:
#| default_exp workflows

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory
import yaml

GitHub Actions, as something you can look at rather than a folder of YAML.

Reading is in the base install and needs pyyaml and nothing else. `parse` returns a row for every file, including the files it could not read. Writing a workflow and everything that calls the GitHub API go through gheasy, which `pullup[cloud]` installs.

`WORKFLOW_DIR` is where GitHub looks. Nothing here reads a workflow from anywhere else.

In [ ]:
#| export
from __future__ import annotations

In [ ]:
#| export
import re

In [ ]:
#| export
from shutil import which

In [ ]:
#| export
from fastcore.all import Path, first

In [ ]:
#| export
from pullup.env import extra

#| export
from pullup.stack import installed

In [ ]:
#| export
WORKFLOW_DIR = '.github/workflows'

In [ ]:
#| export
class WorkflowError(RuntimeError):
    pass

`WorkflowError` is the only exception this module raises. Reading never raises it. `parse` puts the reason on the row's `error` and returns the row. Writing a file, starting a run and pushing a secret raise it.

In [ ]:
#| export
def _gheasy():
    try:
        import gheasy.core as core
        return core
    except ImportError as e:
        raise WorkflowError(f'this needs gheasy: pip install "{extra()}"') from e

`_gheasy` is where every function that needs gheasy asks for it. A missing install fails with the line that installs it, at the call, rather than as an `ImportError` from wherever the import sat.

In [ ]:
#| export
def _yaml():
    """A YAML loader for reading a workflow file.

    gheasy's ruamel instance where gheasy is installed, so a file it wrote round-trips through the
    same parser it was written with. Plain pyyaml otherwise: reading a workflow is worth having in
    the base install, and every part of this module that *writes* one goes through gheasy anyway.
    """
    try:
        from gheasy.workflow import yaml_instance
        return yaml_instance()
    except ImportError: pass
    import yaml
    class _Plain:
        @staticmethod
        def load(text): return yaml.safe_load(text)
    return _Plain()

def levels(jobs):
    "Jobs arranged into rows: everything that can start, then what follows, by longest path."
    known = {j['id'] for j in jobs}
    depth, pending = {}, list(jobs)
    for _ in range(len(jobs) + 1):
        progressed = False
        for job in list(pending):
            needs = [n for n in job['needs'] if n in known]
            if all(n in depth for n in needs):
                depth[job['id']] = 1 + max((depth[n] for n in needs), default=-1)
                pending.remove(job)
                progressed = True
        if not pending or not progressed: break
    cycle = max(depth.values(), default=-1) + 1
    for job in pending: depth[job['id']] = cycle
    rows = {}
    for job in jobs: rows.setdefault(depth[job['id']], []).append(job['id'])
    return [rows[k] for k in sorted(rows)]

`_yaml` returns gheasy's ruamel instance where gheasy is installed, and a pyyaml wrapper otherwise. The two disagree about `on:`. YAML 1.1, which pyyaml implements, reads the bare word `on` as the boolean `True`. YAML 1.2, which ruamel is configured for here, keeps it a string. `parse` looks under both keys, so a workflow reads the same under either parser.

In [ ]:
list(_yaml().load('on: push')), list(yaml.safe_load('on: push'))

(['on'], [True])

`levels` arranges jobs into the rows a graph is drawn in. Row 0 is everything with nothing to wait for. A job sits one row past the deepest job it needs, so an edge never points backwards and never points within a row.

A `needs` naming a job the workflow does not define is dropped, and the job keeps its row. Jobs in a cycle share the row after the last one that resolved, so a workflow that cannot run is still a picture. Each job appears exactly once, and order within a row is the order the jobs came in.

In [ ]:
jobs = [{'id': 'lint', 'needs': []}, {'id': 'test', 'needs': []},
        {'id': 'ship', 'needs': ['lint', 'test']}, {'id': 'notify', 'needs': ['ship']}]
levels(jobs)

[['lint', 'test'], ['ship'], ['notify']]

In [ ]:
#| hide
test_eq(levels([{'id': 'a', 'needs': []}, {'id': 'b', 'needs': ['a']}, {'id': 'c', 'needs': ['a', 'b']}]),
        [['a'], ['b'], ['c']])                                   # the longest path, not the shortest
test_eq(levels([{'id': 'ok', 'needs': []}, {'id': 'x', 'needs': ['y']}, {'id': 'y', 'needs': ['x']}]),
        [['ok'], ['x', 'y']])                                    # a cycle lands in one row of its own
test_eq(levels([]), [])
seen = sum(levels(jobs), [])
test_eq(sorted(seen), sorted(j['id'] for j in jobs))

In [ ]:
#| export
class Workflows:
    "Every workflow in one repository: the files, the graph, the runs, and the secrets."
    def __init__(self, root): self.root = Path(root).expanduser().resolve()
    @property
    def dir(self): return self.root/WORKFLOW_DIR
    def files(self):
        try: return sorted(p for p in self.dir.iterdir() if p.suffix in ('.yml', '.yaml'))
        except OSError: return []
    @staticmethod
    def _triggers(on):
        "`on:` in any of its three shapes: a string, a list, or a mapping with filters."
        if isinstance(on, str): return [{'event': on, 'detail': ''}]
        if isinstance(on, list): return [{'event': str(x), 'detail': ''} for x in on]
        rows = []
        for event, spec in (on or {}).items():
            detail = ''
            if isinstance(spec, dict):
                parts = [f'{k}: {", ".join(map(str, v))}' if isinstance(v, list) else f'{k}: {v}'
                    for k, v in spec.items() if k in ('branches', 'tags', 'paths', 'types', 'cron')]
                detail = ' · '.join(parts)
            elif isinstance(spec, list):
                flat = [', '.join(map(str, x.values())) if isinstance(x, dict) else str(x) for x in spec]
                detail = ', '.join(flat)
            rows.append({'event': str(event), 'detail': detail})
        return rows
    @staticmethod
    def _steps(raw):
        rows = []
        for step in raw or ():
            if not isinstance(step, dict): continue
            uses, run = str(step.get('uses') or ''), str(step.get('run') or '')
            label = str(step.get('name') or '') or uses or (run.splitlines()[0] if run else '') or 'step'
            rows.append({'name': label, 'uses': uses, 'run': run, 'if': str(step.get('if') or '')})
        return rows
    @classmethod
    def _job(cls, job_id, job):
        "One entry of `jobs:`, with `needs` and `environment` in either of the shapes each takes."
        job = job if isinstance(job, dict) else {}
        needs = job.get('needs') or []
        env = job.get('environment')
        return {'id': str(job_id), 'name': str(job.get('name') or job_id),
            'needs': [needs] if isinstance(needs, str) else [str(n) for n in needs],
            'runs_on': str(job.get('runs-on') or job.get('runs_on') or ''),
            'if': str(job.get('if') or ''), 'uses': str(job.get('uses') or ''),
            'environment': str(env.get('name') or '') if isinstance(env, dict) else str(env or ''),
            'steps': cls._steps(job.get('steps'))}
    def parse(self, path):
        "One workflow file as jobs and edges, or as the error that stopped it being read."
        path = Path(path)
        row = {'file': path.name, 'path': str(path), 'name': path.stem, 'triggers': [],
            'jobs': [], 'levels': [], 'dispatchable': False, 'error': ''}
        try: doc = _yaml().load(path.read_text(encoding='utf-8', errors='replace')) or {}
        except (OSError, WorkflowError) as e: return row | {'error': str(e)}
        except Exception as e:
            return row | {'error': f'{type(e).__name__}: {e}'}
        if not isinstance(doc, dict): return row | {'error': 'this file is not a workflow'}
        on = doc.get('on', doc.get(True, {}))
        row['name'] = str(doc.get('name') or path.stem)
        row['triggers'] = self._triggers(on)
        row['dispatchable'] = any(t['event'] == 'workflow_dispatch' for t in row['triggers'])
        jobs = doc.get('jobs') or {}
        row['jobs'] = [self._job(k, v) for k, v in (jobs.items() if isinstance(jobs, dict) else ())]
        row['levels'] = levels(row['jobs'])
        return row
    def parsed(self): return [self.parse(p) for p in self.files()]
    def repo(self):
        "`owner/name` from the origin remote, or `''` when there is no GitHub remote."
        try:
            core = _gheasy()
            owner, name = core._get_repo_slug(str(self.root))
            return f'{owner}/{name}'
        except Exception: return ''
    def _api(self):
        core = _gheasy()
        token = core._resolve_gh_token()
        if not token:
            raise WorkflowError('set GITHUB_TOKEN: put it in the environment store, or export it in the shell this ran from')
        owner, name, api = core._gh_api(token, str(self.root))
        return owner, name, _sync_api(core, owner, name, token, api)
    def status(self):
        "Whether the GitHub half is usable, and if not, which of gheasy, remote or token is missing."
        row = {'gheasy': True, 'dockeasy': installed('dockeasy'), 'repo': '', 'token': False,
            'gh_cli': bool(which('gh')), 'gh_auth': _gh_authed(), 'why': ''}
        try: core = _gheasy()
        except WorkflowError as e: return row | {'gheasy': False, 'why': str(e)}
        row['repo'] = self.repo()
        row['token'] = bool(core._resolve_gh_token())
        if not row['repo']: row['why'] = 'no GitHub remote on this repository'
        elif not row['token']: row['why'] = 'no GITHUB_TOKEN'
        return row
    def runs(self, limit=20, workflow=''):
        "Recent runs, newest first. `workflow` is a file name, matching what `files` returns."
        owner, name, api = self._api()
        kw = {'per_page': max(1, min(int(limit), 100))}
        raw = (api.actions.list_workflow_runs(workflow_id=workflow, **kw) if workflow
            else api.actions.list_workflow_runs_for_repo(**kw))
        return [_run_row(run) for run in (raw.get('workflow_runs') or [])]
    def run_jobs(self, run_id):
        "The jobs of one run, so a red workflow says which box is red."
        owner, name, api = self._api()
        raw = api.actions.list_jobs_for_workflow_run(run_id=int(run_id), per_page=100)
        return [{'id': j.get('id'), 'name': j.get('name') or '', 'status': j.get('status') or '',
            'conclusion': j.get('conclusion') or '', 'url': j.get('html_url') or '',
            'started_at': j.get('started_at') or '', 'completed_at': j.get('completed_at') or '',
            'steps': [{'name': s.get('name') or '', 'status': s.get('status') or '',
                'conclusion': s.get('conclusion') or '', 'number': s.get('number')}
                for s in (j.get('steps') or [])]}
            for j in (raw.get('jobs') or [])]
    def dispatch(self, workflow, ref='', inputs=None):
        "Start a run. The workflow must declare `workflow_dispatch`; GitHub refuses otherwise."
        owner, name, api = self._api()
        if not str(workflow).strip(): raise WorkflowError('choose a workflow to run')
        ref = str(ref or '').strip() or _gheasy()._git_branch(str(self.root))
        api.actions.create_workflow_dispatch(workflow_id=str(workflow), ref=ref, inputs=dict(inputs or {}))
        return {'workflow': str(workflow), 'ref': ref}
    def cancel(self, run_id):
        owner, name, api = self._api()
        api.actions.cancel_workflow_run(run_id=int(run_id))
        return {'cancelled': int(run_id)}
    def config(self):
        "The project's `.gheasy/config.json`, or None when gheasy has not been set up here."
        try: core = _gheasy()
        except WorkflowError: return None
        if not core.cfg_path(str(self.root)).exists(): return None
        return core.GheasyConfig.load(str(self.root)).to_dict()
    def generate(self):
        "The workflow gheasy would write for this project, as text, without writing it."
        core = _gheasy()
        if not core.cfg_path(str(self.root)).exists():
            raise WorkflowError('this project has no .gheasy/config.json to generate from')
        return core.mk_workflow(core.GheasyConfig.load(str(self.root)))
    def write_generated(self):
        "Write it through gheasy's `gh_workflow`, the same call the CLI makes. Never commits."
        core = _gheasy()
        if not core.cfg_path(str(self.root)).exists():
            raise WorkflowError('this project has no .gheasy/config.json to generate from')
        core.gh_workflow(path=str(self.root))
        return {'path': str(self.dir/'gheasy.yml')}
    def detect(self):
        "What this project is, from the files that say so and from dockeasy's `detect_app`."
        root = self.root
        has = lambda *names: any((root/n).exists() for n in names)
        text = ''.join(_read(root/n) for n in ('pyproject.toml', 'requirements.txt')).lower()
        from pullup.project import app_project, fastship_project, nbdev_project
        row = {
            'python': has('pyproject.toml', 'requirements.txt', 'setup.py'),
            'node': has('package.json'), 'rust': has('Cargo.toml'), 'go': has('go.mod'),
            'nbdev': nbdev_project(root), 'fastship': fastship_project(root),
            'app': app_project(root), 'docker': has('Dockerfile'),
            'compose': has('docker-compose.yml', 'compose.yml'),
            'deploy_script': has('deploy.py'), 'pages': has('_quarto.yml', 'nbs/_quarto.yml', 'docs'),
            'web': any(k in text for k in ('python-fasthtml', 'fastapi', 'django', 'flask')),
            'vps': any(k in text for k in ('vpseasy', 'cfeasy', 'hcloud')),
            'publish': any(k in text for k in ('hatchling', 'setuptools', 'flit')),
            'build': '', 'build_why': '',
        }
        try:
            from dockeasy.core import detect_app
            row['build'] = _from_line(detect_app(str(root)))
        except ImportError: row['build_why'] = 'install dockeasy to detect the build'
        except Exception as e: row['build_why'] = f'{type(e).__name__}: {e}'
        return row
    def catalogue(self):
        "The workflows on offer, every one of them gheasy's, each marked with whether this project wants it."
        d = self.detect()
        present = {p.name for p in self.files()}
        rows = [
            {'id': 'ci', 'name': 'Python CI', 'file': 'ci.yml',
                'doc': 'ruff, then pytest, on push and pull request — uv for both.',
                'want': d['python'] and not d['nbdev'], 'why': 'a Python project'},
            {'id': 'nbdev', 'name': 'nbdev CI', 'file': 'ci.yml',
                'doc': 'nbdev_test and a clean-notebook check, which is what breaks in an nbdev repo.',
                'want': d['nbdev'], 'why': 'nbdev project'},
            {'id': 'pypi', 'name': 'PyPI on release', 'file': 'publish.yml',
                'doc': 'Build and publish through PyPI trusted publishing when a release is created.',
                'want': d['publish'] and not d['fastship'], 'why': 'this project builds a distribution'},
            {'id': 'fastship', 'name': 'fastship release', 'file': 'release.yml',
                'doc': 'What `ship-release` pushes a `v*` tag for: build it, write the notes from the '
                'closed issues, publish it through trusted publishing.',
                'want': d['fastship'] and d['publish'], 'why': 'a Python package that is not nbdev'},
            {'id': 'docker', 'name': 'Docker image', 'file': 'docker.yml',
                'doc': 'Build and push to GHCR with semver and sha tags from docker/metadata-action.',
                'want': d['docker'] or bool(d['build']), 'why': 'there is an image to build'},
            {'id': 'deploy', 'name': 'Deploy to a VPS', 'file': 'deploy.yml',
                'doc': 'Everything in env_schema as secrets and variables, the deploy key, then '
                'your deploy script — the shape lego uses.',
                'want': d['deploy_script'] or d['vps'], 'why': 'a deploy script or a VPS dependency'},
            {'id': 'pages', 'name': 'GitHub Pages', 'file': 'pages.yml',
                'doc': 'Build a static site and publish it to Pages.',
                'want': d['pages'] and d['node'], 'why': 'a static site with a node build'},
        ]
        for row in rows:
            row['present'] = row['file'] in present
            row['recommended'] = bool(row['want'])
            row['why'] = row['why'] if row['want'] else ''
        return {'detect': d, 'workflows': rows}
    def _build(self, choice):
        "One catalogue entry as a gheasy `WorkflowDoc`. Nothing here writes anything."
        from gheasy.workflow import Workflow, docker_build_push, pages_deploy, uv_ci
        from .wfbuild import nbdev_job
        cfg = self.config() or {}
        app = str(cfg.get('app') or self.root.name)
        if choice == 'ci': return uv_ci(f'{app} CI')
        if choice == 'fastship': return _fastship(f'{app} release', test_cmd='pytest')
        if choice == 'nbdev':
            wfb = Workflow(f'{app} CI')
            wfb.on.push(branches=['main']).pull_request(branches=['main'])
            nbdev_job(wfb)
            return wfb.build()
        if choice == 'pypi':
            wfb = Workflow(f'{app} publish')
            wfb.on.release(types=['published'])
            wfb.uv_pypi_job(needs=None)
            return wfb.build()
        if choice == 'docker': return docker_build_push(f'{app} image')
        if choice == 'pages': return pages_deploy(f'{app} pages')
        if choice == 'deploy': return self._deploy_workflow(app, cfg)
        raise WorkflowError(f'unknown workflow: {choice}')
    def _deploy_workflow(self, app, cfg):
        "The deploy workflow, with the project's own `env_schema` rendered into its `env:` block."
        from gheasy.workflow import Workflow
        from .wfbuild import deploy_job
        wfb = Workflow(f'{app} deploy')
        wfb.on.push(branches=['main'])
        deploy_job(wfb, 'deploy', {}, cfg.get('env_schema') or {}, app)
        return wfb.build()
    def build(self, spec):
        "A spec from the builder as YAML, composed entirely with gheasy's DSL."
        _gheasy()
        from .wfbuild import compose
        cfg = self.config() or {}
        return compose(spec, schema=self.schema(), app=str(cfg.get('app') or self.root.name))
    def write_spec(self, spec, file=''):
        "Write a built workflow, and keep the spec beside it. Reading YAML back into one is lossy."
        _gheasy()
        from .wfbuild import save_spec
        name = _wf_name(str(file or spec.get('file') or f"{spec.get('name') or 'ci'}.yml"))
        text = self.build(spec)
        path = self.dir/name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding='utf-8')
        save_spec(self.root, name, {**spec, 'file': name})
        return {'path': str(path), 'file': name}
    def spec(self, file):
        "The spec a workflow was built from, plus whether the file on disk still matches it."
        from .wfbuild import load_spec
        spec = load_spec(self.root, file)
        if spec is None: return None
        try: current = (self.dir/file).read_text(encoding='utf-8')
        except OSError: current = ''
        try: stale = bool(current) and current.strip() != self.build(spec).strip()
        except WorkflowError: stale = False
        return {'spec': spec, 'stale': stale}
    def preview(self, choice):
        "The YAML a catalogue entry would write, without writing it."
        _gheasy()
        return self._build(str(choice)).to_yaml()
    def add(self, choice, file=''):
        "Write one catalogue entry into `.github/workflows`. Never overwrites silently."
        _gheasy()
        rows = {r['id']: r for r in self.catalogue()['workflows']}
        row = rows.get(str(choice))
        if row is None: raise WorkflowError(f'unknown workflow: {choice}')
        name = _wf_name(str(file or row['file']))
        path = self.dir/name
        if path.exists(): raise WorkflowError(f'{name} already exists — delete or rename it first')
        path.parent.mkdir(parents=True, exist_ok=True)
        self._build(row['id']).save(path)
        return {'path': str(path), 'workflow': row['id']}
    def docker(self):
        "What dockeasy makes of this project, and whether it already has a Dockerfile."
        row = {'dockerfile': (self.root/'Dockerfile').exists(), 'base': '', 'preview': '', 'why': ''}
        try: from dockeasy.core import detect_app
        except ImportError:
            return row | {'why': f'detecting the build needs dockeasy: pip install "{extra()}"'}
        try: text = str(detect_app(str(self.root)))
        except Exception as e: return row | {'why': f'{type(e).__name__}: {e}'}
        return row | {'preview': text, 'base': _from_line(text)}
    def write_dockerfile(self):
        "Write the Dockerfile dockeasy detected. Refuses to overwrite one that already exists."
        try: from dockeasy.core import detect_app
        except ImportError as e:
            raise WorkflowError(f'writing a Dockerfile needs dockeasy: pip install "{extra()}"') from e
        path = self.root/'Dockerfile'
        if path.exists(): raise WorkflowError('this project already has a Dockerfile')
        text = str(detect_app(str(self.root)))
        path.write_text(text if text.endswith('\n') else text + '\n', encoding='utf-8')
        return {'path': str(path)}
    def schema(self):
        "gheasy's `env_schema`: `{KEY: None}` is a secret, `{KEY: default}` is a variable."
        cfg = self.config() or {}
        return dict(cfg.get('env_schema') or {})
    def remote_env(self):
        "What GitHub already has. Secret *values* are not readable, by GitHub's design."
        owner, name, api = self._api()
        secrets = [{'name': s.get('name'), 'updated_at': s.get('updated_at') or ''}
            for s in (api.actions.list_repo_secrets(per_page=100).get('secrets') or [])]
        variables = [{'name': v.get('name'), 'value': v.get('value') or '', 'updated_at': v.get('updated_at') or ''}
            for v in (api.actions.list_repo_variables(per_page=100).get('variables') or [])]
        return {'secrets': secrets, 'variables': variables}
    def push_env(self, values, keys=(), dry_run=False):
        "Send values up through gheasy's `gh_push_env`, one result per key; undeclared keys are skipped."
        core = _gheasy()
        if not which('gh'):
            raise WorkflowError('pushing secrets uses the GitHub CLI: install `gh` and run `gh auth login`')
        schema = self.schema()
        if not schema:
            raise WorkflowError('declare env_schema in .gheasy/config.json first — '
                'it is what says which keys are secrets and which are variables')
        rows = []
        wanted = keys or values
        for key in wanted:
            value = values.get(key, '')
            if key not in schema:
                rows.append({'key': key, 'ok': True, 'skipped': 'not in env_schema', 'target': ''})
                continue
            target = 'secret' if schema[key] is None else 'variable'
            if not value:
                rows.append({'key': key, 'ok': True, 'skipped': 'no value set', 'target': target})
                continue
            try:
                core.gh_push_env({key: value}, dry_run=dry_run, path=str(self.root))
                rows.append({'key': key, 'ok': True, 'skipped': '', 'target': target})
            except Exception as e:
                rows.append({'key': key, 'ok': False, 'error': f'{type(e).__name__}: {e}', 'target': target})
        return rows

`Workflows` is every workflow in one repository. `files`, `parse`, `parsed`, `detect` and `catalogue` read the working tree and need no token and no network. `runs`, `run_jobs`, `dispatch`, `cancel` and `remote_env` call the GitHub API, and `push_env` shells out to `gh`. This page exercises the first half only.

The example is one workflow of three jobs, in the shapes GitHub allows for each field.

In [ ]:
WF = r'''name: CI
on:
  push:
    branches: [main]
  schedule:
    - cron: '0 4 * * 1'
  workflow_dispatch:
jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: ruff check .
  test:
    name: pytest
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: run the suite
        run: |
          uv sync
          pytest -q
  ship:
    needs: [lint, test]
    if: github.ref == 'refs/heads/main'
    environment: pypi
    runs-on: ubuntu-latest
    steps:
      - run: echo ship
'''

In [ ]:
#| hide
tmp = TemporaryDirectory(); repo = Path(tmp.name); wfs = repo/WORKFLOW_DIR
wfs.mkdir(parents=True)
(wfs/'ci.yml').write_text(WF)
(wfs/'docs.yml').write_text('name: Docs\non: workflow_dispatch\njobs: {}\n')
(wfs/'notes.md').write_text('# what these do\n')

16

`files` returns the `.yml` and `.yaml` files of `.github/workflows`, sorted by name. Anything else in the folder is not a workflow to GitHub and is not one here. A repository without the folder returns `[]`, which is not an error.

In [ ]:
w = Workflows(repo)
[p.name for p in w.files()]

['ci.yml', 'docs.yml']

`parse` reads one file into the row a panel draws. It never raises. A file that cannot be read, cannot be parsed, or is not a mapping comes back as the same row with `error` set and the rest empty, so a folder with one broken file still lists.

`name` is the workflow's `name:`, and the file stem where the workflow has none. `dispatchable` says whether `workflow_dispatch` is among the triggers, which is what decides whether a run can be started from a button.

In [ ]:
row = w.parse(wfs/'ci.yml')
row['name'], row['dispatchable'], row['levels']

('CI', True, [['lint', 'test'], ['ship']])

In [ ]:
[(r['name'], r['dispatchable'], r['error']) for r in w.parsed()]

[('CI', True, ''), ('Docs', True, '')]

`on:` takes three shapes and `_triggers` flattens all of them into an `event` and a `detail`. A string is one event. A list is one event each. A mapping carries filters, and `detail` renders `branches`, `tags`, `paths`, `types` and `cron`, dropping every other key. `detail` is display text. Nothing matches on it.

In [ ]:
row['triggers']

[{'event': 'push', 'detail': 'branches: main'},
 {'event': 'schedule', 'detail': '0 4 * * 1'},
 {'event': 'workflow_dispatch', 'detail': ''}]

In [ ]:
#| hide
test_eq(Workflows._triggers('push'), [{'event': 'push', 'detail': ''}])
test_eq([t['event'] for t in Workflows._triggers(['push', 'fork'])], ['push', 'fork'])
detail = Workflows._triggers({'pull_request': {'branches': ['main', 'dev'], 'types': ['opened'],
                                               'secrets': 'inherit'}})[0]['detail']
assert 'branches: main, dev' in detail and 'types: opened' in detail, detail
assert 'secrets' not in detail, 'a key that is not a filter is not shown'
test_eq(Workflows._triggers({'schedule': [{'cron': '0 4 * * 1'}]}),
        [{'event': 'schedule', 'detail': '0 4 * * 1'}])           # not the mapping's repr
test_eq(Workflows._triggers(None), [])

`_job` normalises one entry of `jobs:`. A `needs` written as a bare string becomes a list of one. An `environment` written as a mapping becomes its `name`. A job whose value is not a mapping keeps its id and gets empty everything else. Every field is a `str` or a list of `str`, never `None`, because these go into a template as they are.

A step's label is its `name`, then its `uses`, then the first line of its `run`, then `step`. `run` keeps all of its text. Only the label is cut to one line.

In [ ]:
[(j['id'], j['name'], j['needs'], j['environment']) for j in row['jobs']]

[('lint', 'lint', [], ''),
 ('test', 'pytest', [], ''),
 ('ship', 'ship', ['lint', 'test'], 'pypi')]

In [ ]:
[(s['name'], s['uses']) for s in row['jobs'][1]['steps']]

[('actions/checkout@v4', 'actions/checkout@v4'), ('run the suite', '')]

In [ ]:
#| hide
test_eq(Workflows._job('ship', {'needs': 'build', 'environment': {'name': 'pypi', 'url': 'https://pypi.org'}}),
        {'id': 'ship', 'name': 'ship', 'needs': ['build'], 'runs_on': '', 'if': '', 'uses': '',
         'environment': 'pypi', 'steps': []})
test_eq(Workflows._job('broken', 'not a mapping')['id'], 'broken')
test_eq([s['name'] for s in Workflows._steps([{'run': 'make all\nmake test'}, {'uses': 'x@v1'}, {}, 'nope'])],
        ['make all', 'x@v1', 'step'])
test_eq(Workflows._steps([{'run': 'make all\nmake test'}])[0]['run'], 'make all\nmake test')
test_eq(row['jobs'][1]['runs_on'], 'ubuntu-latest')

In [ ]:
#| hide
bad = w.parse(wfs/'gone.yml')
assert bad['error'] and bad['file'] == 'gone.yml', 'a file that is not there is a row, not an exception'
test_eq((bad['jobs'], bad['triggers'], bad['levels']), ([], [], []))

The GitHub half needs a token and the network, so nothing on this page calls it. `status` is what a panel asks first: whether gheasy is importable, what the origin remote resolves to, whether a token is set, and whether `gh` is installed and logged in. `why` says which of gheasy, the remote and the token is missing, and is empty once all three are there. The two `gh` fields never appear in `why`, because only `push_env` needs the CLI.

Every method that calls GitHub goes through `_api`, which raises `WorkflowError` for a missing gheasy or a missing token before it makes a request. `_api` rebuilds gheasy's client synchronous through `_sync_api`, because ghapi 2 defaults to async and nothing here awaits.

`add` and `write_spec` write into `.github/workflows` and commit nothing. `add` refuses a name that already exists. `write_spec` overwrites the file it names, and keeps the spec beside it, because reading a workflow back into a spec is lossy. `preview` returns the YAML a catalogue entry would write and `generate` returns what gheasy would write from `.gheasy/config.json`. Neither writes anything.

`push_env` sends only the keys `env_schema` declares and reports a skipped row for the rest. `env_schema` also decides how a key goes up: `None` is a secret, any other default is a variable. Secret values are never readable, by GitHub's design, so `remote_env` returns secret names and the time each changed.

In [ ]:
#| export
def _fastship(name, **kw):
    "gheasy's fastship release recipe, which gheasy releases before 0.0.8 do not carry."
    try: from gheasy.workflow import fastship_release
    except ImportError as e:
        raise WorkflowError('this gheasy has no fastship release recipe: pip install -U gheasy') from e
    return fastship_release(name, **kw)

`_fastship` is a soft import of a recipe gheasy gained in 0.0.8. An older gheasy raises `WorkflowError` naming the upgrade.

In [ ]:
#| export
def _sync_api(core, owner, name, token, api):
    "gheasy's client rebuilt synchronous: ghapi 2 is async by default and nothing here awaits."
    from inspect import signature
    GhApi = getattr(core, 'GhApi', None)
    if GhApi is None or 'sync' not in signature(GhApi).parameters: return api
    return GhApi(owner=owner, repo=name, token=token, sync=True)

In [ ]:
#| export
def _gh_authed():
    "Whether the GitHub CLI is logged in, which only `gh auth status` knows."
    if not which('gh'): return False
    import subprocess
    try:
        return subprocess.run(['gh', 'auth', 'status'], capture_output=True, timeout=8).returncode == 0
    except (OSError, subprocess.SubprocessError): return False

`_gh_authed` runs `gh auth status`, which is the only thing that knows whether the CLI is logged in. A missing `gh`, a timeout or a non-zero exit all read as not logged in.

In [ ]:
#| export
def _run_row(run):
    "One run as the panel shows it; the title falls back to the head commit's first line."
    head = run.get('head_commit') if isinstance(run.get('head_commit'), dict) else {}
    return {'id': run.get('id'), 'name': run.get('name') or '', 'file': _wf_file(run),
        'number': run.get('run_number'), 'status': run.get('status') or '',
        'conclusion': run.get('conclusion') or '', 'branch': run.get('head_branch') or '',
        'event': run.get('event') or '', 'created_at': run.get('created_at') or '',
        'updated_at': run.get('updated_at') or '', 'url': run.get('html_url') or '',
        'actor': ((run.get('actor') or {}).get('login') or ''),
        'title': str(run.get('display_title') or (head.get('message') or '')).split('\n')[0]}

In [ ]:
#| export
def _read(path):
    "A file's text, or `''` when it is not there, which is what a detector wants of a missing file."
    try: return Path(path).read_text(encoding='utf-8', errors='replace')
    except OSError: return ''

In [ ]:
#| export
def _wf_file(run):
    "The file a run came from; GitHub reports it as a path on the workflow, not a name."
    return re.sub(r'^\.github/workflows/', '', run.get('path') or '')

In [ ]:
#| export
def _from_line(text):
    "The `FROM` line of a Dockerfile, which is what a panel shows of a detected build."
    return first(str(text).splitlines(), lambda l: l.startswith('FROM ')) or ''

In [ ]:
#| export
def _wf_name(name):
    "A workflow file name, or the error that says why it is not one: no directories, no other suffix."
    if '/' in name or not name.endswith(('.yml', '.yaml')):
        raise WorkflowError(f'not a workflow file name: {name}')
    return name

`_wf_name` is the check every write goes through. A name with a directory in it, or with any suffix other than `.yml` or `.yaml`, is refused. Nothing outside `.github/workflows` is written by this module.

In [ ]:
test_fail(lambda: _wf_name('.github/workflows/ci.yml'), contains='not a workflow file name')
test_fail(lambda: _wf_name('Makefile'), contains='not a workflow file name')
test_eq(_wf_name('ci.yaml'), 'ci.yaml')
test_eq(_from_line('# built by dockeasy\nFROM python:3.12-slim\nRUN uv sync'), 'FROM python:3.12-slim')
test_eq(_from_line(''), '')
test_eq(_read(repo/'nowhere.toml'), '')              # a missing file reads as no text, not as an error

`_run_row` is one run of the API's `workflow_runs` as the panel shows it. Every text field is a `str`, and a key GitHub omits reads as `''` rather than `None`. `id` and `number` come through as GitHub sent them.

`title` is the run's `display_title`, and the first line of the head commit's message where there is no display title. `file` is `path` with `.github/workflows/` off the front, so it matches what `files` returns and a run traces back to the file it came from.

In [ ]:
run = {'id': 178, 'name': 'CI', 'path': '.github/workflows/ci.yml', 'run_number': 12,
       'status': 'completed', 'conclusion': 'failure', 'head_branch': 'main', 'event': 'push',
       'html_url': 'https://github.com/o/r/actions/runs/178', 'actor': {'login': 'jph'},
       'head_commit': {'message': 'drop the retry\n\nit never fired'}}
_run_row(run)

{'id': 178,
 'name': 'CI',
 'file': 'ci.yml',
 'number': 12,
 'status': 'completed',
 'conclusion': 'failure',
 'branch': 'main',
 'event': 'push',
 'created_at': '',
 'updated_at': '',
 'url': 'https://github.com/o/r/actions/runs/178',
 'actor': 'jph',
 'title': 'drop the retry'}

In [ ]:
#| hide
test_eq(_run_row(run)['title'], 'drop the retry')
test_eq(_run_row(run | {'display_title': 'run 12'})['title'], 'run 12')
test_eq(_run_row(run | {'head_commit': None})['title'], '')
test_eq(_run_row({})['file'], '')
assert all(_run_row({})[k] == '' for k in ('name', 'status', 'conclusion', 'branch', 'url', 'title'))
test_eq(_wf_file({'path': 'ci.yml'}), 'ci.yml')      # already a name, left alone

`detect` is what the working tree says about the project. Every value is a file existing or a name appearing in `pyproject.toml` or `requirements.txt`. `build` is dockeasy's `detect_app` cut down to its `FROM` line, and `build_why` says why it is empty: no dockeasy, or the error `detect_app` raised.

In [ ]:
(repo/'pyproject.toml').write_text('[project]\nname = "demo"\ndependencies = ["python-fasthtml"]\n'
                                   '[build-system]\nrequires = ["hatchling"]\n')
det = w.detect()
{k: v for k, v in det.items() if v}

{'python': True,
 'fastship': True,
 'web': True,
 'publish': True,
 'build': 'FROM ghcr.io/astral-sh/uv:python3.13-bookworm AS builder'}

`catalogue` is gheasy's workflows, each marked for this project. `recommended` comes from `detect`. `present` is whether a file of that name is already in `.github/workflows`, so the two entries that write `ci.yml`, `ci` and `nbdev`, both read as present here. `why` is the reason for a recommendation and is empty on entries that are not recommended.

In [ ]:
[(r['id'], r['recommended'], r['present'], r['why']) for r in w.catalogue()['workflows']]

[('ci', True, True, 'a Python project'),
 ('nbdev', False, True, ''),
 ('pypi', False, False, ''),
 ('fastship', True, False, 'a Python package that is not nbdev'),
 ('docker', True, False, 'there is an image to build'),
 ('deploy', False, False, ''),
 ('pages', False, False, '')]

In [ ]:
#| hide
rows = w.catalogue()['workflows']
test_eq({r['id'] for r in rows if r['present']}, {'ci', 'nbdev'})   # one file, two entries that write it
assert not any(r['why'] for r in rows if not r['recommended'])
test_eq(w.schema(), {})                                            # no .gheasy/config.json, nothing declared
test_is(w.config(), None)
test_fail(lambda: w.generate(), contains='no .gheasy/config.json')

In [ ]:
#| hide
tmp.cleanup()

In [ ]:
#| hide
# `_yaml_instance` was a private import: a request for a public name in gheasy, not a fix here.
# gheasy 0.0.10 opened it, so a workflow pullup reads round-trips through the parser that wrote it.
import gheasy.workflow as _w
assert 'yaml_instance' in _w.__all__
test_eq(_yaml().load('on:\n  push:\n    branches: [main]\n')['on']['push']['branches'], ['main'])